# Notebook 4: Statistical Analysis of Experiment Results

This notebook loads the data from a completed experiment run (identified by its `experiment_id`) and performs a comprehensive statistical analysis. It replicates the functionality of the `scripts/export_reporting.py` script in an interactive format.

**Analysis Steps:**
1. **Load Data**: Fetch the raw logs from the `policy_evaluation_log` table for the specified experiment.
2. **Compute Metrics**: Calculate key performance indicators (KPIs) like Average Process Time (APT), quantiles (Q90), Conditional Value-at-Risk (CVaR), and Miss Rate for various policy and scenario groups.
3. **Significance Testing**: Perform Welch's t-tests to compare the performance of different policies and apply the Holm-Bonferroni correction for multiple comparisons.
4. **Component Decomposition**: Isolate whether observed APT/CVaR differences between the `hmean` and `cvar` policy groups are attributable to the CVaR objective (+ hard constraints + hysteresis) or to personalization, using the `cvar-nopers` ablation variant as an intermediate policy.
5. **Generate Summaries**: Create summary tables that are ready for reporting.

**Instructions:**
- Paste the `experiment_id` you obtained from the previous notebook (`03_offline_replay...`) into the designated cell below.

> **Note on interpretation**: As shown in Section 12.2 of the project README, the nominal-condition increase in APT and CVaR from Hybrid-Mean to Hybrid-CVaR is overwhelmingly (96–101%) attributable to passenger-level personalization, not to the CVaR objective itself. Section 4A below reproduces this decomposition directly from the logged experiment data (via `analyzer._summarize_metrics`, the same routine used throughout this pipeline) so it can be verified on any experiment run, not just the one reported in the manuscript.

> **Note on policy naming**: `policy_group` in the logged data uses this codebase's internal codes, not the manuscript's display names. The mapping used throughout this notebook is: `baseline` = Baseline, `hmean` = Hybrid-Mean, `cvar-nopers` = CVaR-NoPersonalization, `cvar-nohys` = CVaR-NoHysteresis, `cvar` = Hybrid-CVaR (proposed).

In [ ]:
import sys
import os
import pandas as pd
import numpy as np

# Add project root to path
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.append(project_root)

from src.analytics import OfflineReplayAnalyzer, TableExporter
from pathlib import Path

# Configure pandas for better display
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 120)

### 1. Set Experiment ID and Initialize Analyzer

👇 **ACTION REQUIRED**: Paste your `experiment_id` from the previous notebook here.

In [ ]:
# PASTE YOUR EXPERIMENT ID HERE
experiment_id = "<PASTE YOUR EXPERIMENT ID HERE>"

if "<PASTE" in experiment_id:
    print("⚠️ Please replace the placeholder with your actual experiment ID.")
else:
    print(f"Analyzing Experiment ID: {experiment_id}")
    analyzer = OfflineReplayAnalyzer(experiment_id)


### 2. Load and Prepare Data

In [ ]:
try:
    analyzer.load_data()
    analyzer.prepare_data()
    print("Data loaded and prepared successfully.")
    display(analyzer.df_log.head())
    print(f"\nTotal log entries: {len(analyzer.df_log)}")
    print(f"Policy groups present: {sorted(analyzer.df_log['policy_group'].unique().tolist())}")
except Exception as e:
    print(f"❌ Failed to load data. Have you pasted the correct experiment ID? Error: {e}")

### 3. Analysis of Q4 (Parking Scenario)

The Q4 scenario is often used for primary policy comparisons due to its complexity (involving parking occupancy). Here we compute the main performance metrics for each policy group within this scenario.

In [ ]:
df_q4_metrics = analyzer._analyze_q4_performance()
print("--- Q4 Performance Metrics by Policy ---")
display(df_q4_metrics)

### 4. Statistical Significance Testing (E1-E5)

This performs Welch's t-tests to compare key policy pairs and calculates Cohen's d for effect size. The p-values are then adjusted using the Holm-Bonferroni method to control the family-wise error rate.

Note that `E2` here (`hmean` vs `cvar`) reports the *aggregate* Hybrid-Mean → Hybrid-CVaR effect without separating the contribution of personalization from the CVaR objective, hard constraints, and hysteresis. Section 4A below performs that separation.

In [ ]:
df_tests, p_values = analyzer._run_e1_e5_tests()
analyzer.results['e1_e5_tests'] = df_tests # Store before correction
analyzer._apply_holm_bonferroni(p_values)

print("--- Statistical Test Results (E1-E5 Experiments) ---")
display(analyzer.results['e1_e5_tests'])

### 4A. Component Decomposition Analysis (Personalization vs. CVaR Objective)

**Why this section exists:** the `cvar` (Hybrid-CVaR) policy group differs from `hmean` (Hybrid-Mean) in four components simultaneously — the CVaR objective, hard constraints, hysteresis, and personalization (see the Policy Configuration Matrix in the manuscript, Table 4). A naive `hmean` vs. `cvar` comparison (as in `E2` above) therefore cannot attribute the observed APT/CVaR₀.₉ increase to any single mechanism.

**Method:** we decompose the `hmean` → `cvar` transition into two sequential steps using the `cvar-nopers` ablation variant already logged by `ExperimentRunner`:

1. **`hmean` → `cvar-nopers`**: introduces the CVaR objective, hard constraints, and hysteresis, *without* personalization.
2. **`cvar-nopers` → `cvar`**: adds personalization on top of step 1.

The share of the total `hmean` → `cvar` difference attributable to personalization is then:

$$\text{Personalization Share} = \frac{\Delta_{\text{step 2}}}{\Delta_{\text{step 1}} + \Delta_{\text{step 2}}}$$

computed separately for APT_mean and CVaR₀.₉. Each policy-group slice is scored with `analyzer._summarize_metrics(...)`, the same routine `_analyze_q4_performance` and `_analyze_e6_performance` use, so results here are directly comparable to Sections 3 and beyond. This reproduces the decomposition reported in the manuscript (Table 13) and in Section 12.2 of the project README, computed here from this experiment's own logged data rather than the manuscript's fixed numbers.

In [ ]:
# (policy_group code, display label), in decomposition order
DECOMPOSITION_STEPS = [
    ('hmean', 'Hybrid-Mean'),
    ('cvar-nopers', 'CVaR-NoPersonalization'),
    ('cvar', 'Hybrid-CVaR'),
]
SCENARIOS = ['Q1', 'Q2', 'Q3', 'Q4']


def _safe_share(numerator, denominator):
    if numerator is None or denominator is None:
        return np.nan
    if pd.isna(numerator) or pd.isna(denominator) or denominator == 0:
        return np.nan
    return numerator / denominator * 100.0


def compute_component_decomposition(analyzer, steps=DECOMPOSITION_STEPS, scenarios=SCENARIOS):
    """
    Decomposes the 'hmean' -> 'cvar' (Hybrid-Mean -> Hybrid-CVaR) difference in APT_mean
    and CVaR_0.9 into:
      step 1: 'hmean' -> 'cvar-nopers'   (CVaR objective + hard constraints + hysteresis)
      step 2: 'cvar-nopers' -> 'cvar'    (personalization)

    Reuses analyzer._summarize_metrics(...) so results are computed with exactly the same
    logic (including the compute_cvar helper) as the rest of the pipeline, per
    (scenario, policy_group).
    """
    df = analyzer.df_log
    (code1, label1), (code2, label2), (code3, label3) = steps
    rows = []
    missing_groups = set()

    for scenario in scenarios:
        df_scenario = df[df['scenario'] == scenario]
        if df_scenario.empty:
            continue

        metrics = {}
        for code, _label in steps:
            df_sub = df_scenario[df_scenario['policy_group'] == code]
            if df_sub.empty:
                missing_groups.add(code)
                metrics[code] = {'APT_mean': np.nan, 'CVaR_0.9': np.nan}
            else:
                metrics[code] = analyzer._summarize_metrics(df_sub)

        apt1, apt2, apt3 = (metrics[code1]['APT_mean'], metrics[code2]['APT_mean'], metrics[code3]['APT_mean'])
        cvar1, cvar2, cvar3 = (metrics[code1]['CVaR_0.9'], metrics[code2]['CVaR_0.9'], metrics[code3]['CVaR_0.9'])

        d_apt_step1, d_apt_step2 = apt2 - apt1, apt3 - apt2
        d_apt_total = apt3 - apt1
        d_cvar_step1, d_cvar_step2 = cvar2 - cvar1, cvar3 - cvar2
        d_cvar_total = cvar3 - cvar1

        rows.append({
            'scenario': scenario,
            f'apt_{code1}': apt1, f'apt_{code2}': apt2, f'apt_{code3}': apt3,
            'delta_apt_cvar_obj_constraints_hysteresis': d_apt_step1,
            'delta_apt_personalization': d_apt_step2,
            'delta_apt_total': d_apt_total,
            'personalization_share_apt_pct': _safe_share(d_apt_step2, d_apt_total),
            f'cvar_{code1}': cvar1, f'cvar_{code2}': cvar2, f'cvar_{code3}': cvar3,
            'delta_cvar_cvar_obj_constraints_hysteresis': d_cvar_step1,
            'delta_cvar_personalization': d_cvar_step2,
            'delta_cvar_total': d_cvar_total,
            'personalization_share_cvar_pct': _safe_share(d_cvar_step2, d_cvar_total),
        })

    if missing_groups:
        print(f"⚠️ Policy group(s) not found in this experiment's log for one or more scenarios "
              f"(rows involving these will show NaN): {sorted(missing_groups)}")

    return pd.DataFrame(rows)


df_decomposition = compute_component_decomposition(analyzer)
analyzer.results['component_decomposition'] = df_decomposition

print("--- Component Decomposition: hmean -> cvar (Hybrid-Mean -> Hybrid-CVaR) ---")
display(df_decomposition[[
    'scenario',
    'delta_apt_cvar_obj_constraints_hysteresis', 'delta_apt_personalization',
    'personalization_share_apt_pct',
    'delta_cvar_cvar_obj_constraints_hysteresis', 'delta_cvar_personalization',
    'personalization_share_cvar_pct',
]])

**Reading this table:** for each scenario, `personalization_share_*_pct` close to 100% indicates that essentially all of the `hmean` → `cvar` increase in that metric is attributable to personalization, not to the CVaR objective, hard constraints, or hysteresis. Values reported in the manuscript (Table 13) range from 96–101% across Q1–Q4 for both APT and CVaR₀.₉. Substantially lower shares on your own experiment run would indicate that this experiment's calibration, sample size, or random seed diverges from the reported results and warrants investigation before drawing conclusions from it.

The two contributions are visualized below (mirrors the manuscript's Figure 7, generated here per-scenario rather than for Q4 only).

In [ ]:
import matplotlib.pyplot as plt

code1, label1 = DECOMPOSITION_STEPS[0]
code2, label2 = DECOMPOSITION_STEPS[1]
code3, label3 = DECOMPOSITION_STEPS[2]

n_panels = len(df_decomposition)
if n_panels == 0:
    print("⚠️ No scenarios available for plotting (df_decomposition is empty).")
else:
    fig, axes = plt.subplots(1, n_panels, figsize=(4.5 * n_panels, 4.5), sharey=False)
    if n_panels == 1:
        axes = [axes]

    for ax, (_, row) in zip(axes, df_decomposition.iterrows()):
        labels = [label1, '+ CVaR obj.\n+ constraints\n+ hysteresis', f'+ Personalization\n({label3})']
        values = [row[f'apt_{code1}'], row[f'apt_{code2}'], row[f'apt_{code3}']]
        ax.bar(labels, values, color=['#8c9aa8', '#2f8f7f', '#1a5c4f'])
        ax.set_title(f"{row['scenario']}")
        ax.set_ylabel('APT (min)')
        for i, v in enumerate(values):
            if pd.notna(v):
                ax.text(i, v, f"{v:.2f}", ha='center', va='bottom', fontsize=9)

    fig.suptitle('Component Decomposition of the Mean-Time Increase (Hybrid-Mean → Hybrid-CVaR)', y=1.05)
    fig.tight_layout()
    plt.show()

### 5. Overall Policy Summary

This table provides a high-level summary of each policy's performance across all scenarios, including the `switch_rate`, which measures recommendation stability.

In [ ]:
df_policy_summary = analyzer._generate_policy_summary()
print("--- Overall Policy Summary (All Scenarios) ---")
display(df_policy_summary.sort_values('APT_mean'))

### 5A. Switch-Rate Interaction Check (Hysteresis × Personalization)

Section 12.3 of the project README reports that recommendation stability depends on the **joint** operation of hysteresis and personalization rather than on either mechanism alone. The cell below reproduces the four nested switch-rate comparisons directly from `df_policy_summary` (aggregated across all scenarios, using `policy_group` codes), when the corresponding ablation policies (`cvar-nopers`, `cvar-nohys`) are present in this experiment run.

Note: `switch_rate` here is computed by `analyzer._generate_policy_summary()` from actual passenger-level trajectories (grouped by `scenario`, `u`, `user_segment`, and `occupancy` where available) rather than re-derived in this notebook, so it reflects exactly what is exported in `policy_summary.csv`.

In [ ]:
def _switch_rate(policy_group_code):
    row = df_policy_summary[df_policy_summary['policy_group'] == policy_group_code]
    if row.empty or 'switch_rate' not in row.columns:
        return np.nan
    return row['switch_rate'].iloc[0]

sr_hmean = _switch_rate('hmean')
sr_no_pers = _switch_rate('cvar-nopers')
sr_no_hyst = _switch_rate('cvar-nohys')
sr_cvar = _switch_rate('cvar')

df_interaction = pd.DataFrame([
    {'transition': 'hmean -> cvar-nopers (adds CVaR obj. + hysteresis, no personalization)',
     'from_rate': sr_hmean, 'to_rate': sr_no_pers,
     'interpretation': 'Hysteresis alone (no personalization)'},
    {'transition': 'cvar-nopers -> cvar (adds personalization)',
     'from_rate': sr_no_pers, 'to_rate': sr_cvar,
     'interpretation': 'Personalization completes the stabilizing effect'},
    {'transition': 'hmean -> cvar-nohys (adds CVaR obj. + personalization, no hysteresis)',
     'from_rate': sr_hmean, 'to_rate': sr_no_hyst,
     'interpretation': 'Personalization alone (no hysteresis)'},
    {'transition': 'cvar-nohys -> cvar (adds hysteresis)',
     'from_rate': sr_no_hyst, 'to_rate': sr_cvar,
     'interpretation': 'Hysteresis completes the stabilizing effect'},
])
df_interaction['delta'] = df_interaction['to_rate'] - df_interaction['from_rate']

analyzer.results['switch_rate_interaction'] = df_interaction

print("--- Switch-Rate Interaction: Hysteresis x Personalization (aggregate, all scenarios) ---")
display(df_interaction)

if df_interaction[['from_rate', 'to_rate']].isna().any().any():
    print("⚠️ One or more policy groups ('hmean', 'cvar-nopers', 'cvar-nohys', 'cvar') were not "
          "found in this experiment's log — rows involving them show NaN. Re-run with an "
          "experiment configuration that logs all four groups to complete this check.")

### 6. Export All Results

Finally, we can run the full pipeline function to populate all result tables and export them to an Excel file and individual CSVs in the `output/` directory. The component decomposition and switch-rate interaction tables computed above are included in the export as `component_decomposition.csv` and `switch_rate_interaction.csv`.

In [ ]:
print("Running the full analysis and export pipeline...")
output_dir = Path(project_root) / 'output'
analyzer.run_full_pipeline(output_dir=str(output_dir))

# run_full_pipeline() recomputes analyzer.results from scratch and does not yet natively
# include the two tables computed interactively in this notebook (Sections 4A and 5A), so
# we export them separately here to keep output/ complete and consistent with the README.
output_dir.mkdir(parents=True, exist_ok=True)
if 'component_decomposition' in analyzer.results:
    analyzer.results['component_decomposition'].to_csv(output_dir / 'component_decomposition.csv', index=False)
    print(f"Saved: {output_dir / 'component_decomposition.csv'}")
if 'switch_rate_interaction' in analyzer.results:
    analyzer.results['switch_rate_interaction'].to_csv(output_dir / 'switch_rate_interaction.csv', index=False)
    print(f"Saved: {output_dir / 'switch_rate_interaction.csv'}")

print(f"\n✅ All results have been exported to the '{output_dir.name}' directory.")